##1. Downloading the images

In [1]:
!pip install icrawler

In [3]:
#Downloading the images

from icrawler.builtin import BingImageCrawler

chicken_crawler = BingImageCrawler(
    downloader_threads=4,
    storage={'root_dir': 'data/chicken'}
)

chicken_crawler.crawl(
    keyword='chicken bird natural photo',
    max_num=150,
    min_size=(200, 200)
)

duck_crawler = BingImageCrawler(
    downloader_threads=4,
    storage={'root_dir': 'data/duck'}
)

duck_crawler.crawl(
    keyword='duck bird natural photo',
    max_num=150,
    min_size=(200, 200)
)

In [8]:
#removing corrupted images

from PIL import Image
import os

def remove_corrupted_images(folder):
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        try:
            img = Image.open(path)
            img.verify()
        except:
            os.remove(path)

remove_corrupted_images("data/chicken")
remove_corrupted_images("data/duck")

##2. Fine-tuning the model

In [9]:
!pip install scikit-learn

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os

In [11]:
#transformation pipeline for images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [12]:
dataset = datasets.ImageFolder(root='data', transform=transform)

# Train/Test split (80/20)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

class_names = dataset.classes
print("Classes:", class_names)

Classes: ['chicken', 'duck']


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)

# Replace final layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]


In [14]:
#enable fine-tuning all parameters of the pre-trained model
for param in model.parameters():
    param.requires_grad = True

In [15]:
#initialize loss function and the optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [16]:
#Run training epochs

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/5], Loss: 0.3780
Epoch [2/5], Loss: 0.0302
Epoch [3/5], Loss: 0.0082
Epoch [4/5], Loss: 0.0048
Epoch [5/5], Loss: 0.0143


##3. Evaluation and classiication report

In [17]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("Confusion Matrix:\n")
print(confusion_matrix(all_labels, all_preds))

Classification Report:

              precision    recall  f1-score   support

     chicken       1.00      1.00      1.00        18
        duck       1.00      1.00      1.00        18

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

Confusion Matrix:

[[18  0]
 [ 0 18]]
